# Boundary Conditions and Convergence

Use boundary data and resolution changes to test whether wave-equation errors decrease as expected.

Navigation: [Index](../index.ipynb) | Previous: [Method of Lines and Runge-Kutta Methods](method_of_lines_and_rk.ipynb) | Next: [Finite-Difference Playground](finite_difference_playground.ipynb)


## Generate, Run, and Compare Diagnostics
The Cartesian wave project uses periodic boundary handling in its compact test case. Running with convergence factor 2 refines the grid and should reduce the diagnostic error.

## Import Boundary-Test Execution Helpers

These standard-library tools run commands, manage temporary project directories, and clean command output.


In [ ]:
from pathlib import Path
import re, shutil, subprocess, sys, tempfile


def clean_command_output(text):
    cleaned = re.sub(r"\x1b\[[0-?]*[ -/]*[@-~]", "", text or "")
    return cleaned.replace(str(WORKSPACE), "<workspace>")


def run_command(args, cwd, timeout):
    try:
        result = subprocess.run(
            args,
            cwd=cwd,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            check=True,
            timeout=timeout,
        )
    except FileNotFoundError as exc:
        raise RuntimeError(f"Required command is missing: {args[0]}") from exc
    except subprocess.CalledProcessError as exc:
        print(clean_command_output(exc.stdout))
        raise RuntimeError(f"Command failed: {' '.join(map(str, args))}") from exc
    return clean_command_output(result.stdout)


def require_toolchain():
    if shutil.which("make") is None:
        raise RuntimeError(
            "This notebook requires make to build the generated project."
        )
    if not any(shutil.which(name) for name in ["cc", "gcc", "clang"]):
        raise RuntimeError(
            "This notebook requires a C compiler such as cc, gcc, or clang."
        )


## Create a Boundary-Test Workspace

The workspace keeps generated files separate from the tutorial source tree.


In [ ]:
PROJECT_NAME = "wave_equation_cartesian"
workspace_manager = tempfile.TemporaryDirectory(
    prefix="nrpy_tutorial_cartesian_", dir=Path.cwd()
)
WORKSPACE = Path(workspace_manager.name)
PROJECT_DIR = WORKSPACE / "project" / PROJECT_NAME


## Generate the Boundary-Test Project

This command invokes the same module a learner can run from a terminal and then verifies that the project directory exists.


In [ ]:
command = [sys.executable, "-m", "nrpy.examples.wave_equation_cartesian"]
print("generator command: python -m nrpy.examples.wave_equation_cartesian")
output = run_command(command, WORKSPACE, timeout=300)
for line in output.splitlines():
    if line.strip():
        print(line.rstrip())
if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(PROJECT_DIR)
print("project path: project/wave_equation_cartesian")


## Shorten the Boundary-Test Runtime

Only runtime values are changed so the notebook run finishes quickly.


In [ ]:
parfile = PROJECT_DIR / "wave_equation_cartesian.par"
par_text = parfile.read_text(encoding="utf-8")
par_text = par_text.replace("t_final = 8.0", "t_final = 0.3125")
par_text = par_text.replace(
    "diagnostics_output_every = 0.2", "diagnostics_output_every = 0.15625"
)
par_text = par_text.replace(
    "output_progress_every = 1", "output_progress_every = 1000000"
)
parfile.write_text(par_text, encoding="utf-8")
print("--- runtime wave_equation_cartesian.par ---")
print(parfile.read_text(encoding="utf-8", errors="replace"))


## Build the Boundary-Test Executable

The build step compiles generated C after checking that external build tools are available.


In [ ]:
require_toolchain()
build_output = run_command(["make", "-j2"], PROJECT_DIR, timeout=300)
print("build completed")
print("compiler output line count:", len(build_output.splitlines()))


## Run the Coarse Boundary Test

The default run supplies the coarse diagnostic comparison.


In [ ]:
default_output = run_command([f"./{PROJECT_NAME}"], PROJECT_DIR, timeout=90)
print("default run completed")
for line in default_output.splitlines()[:8]:
    if line.strip():
        print(line.rstrip())


## Run the Refined Boundary Test

The convergence-factor run uses the same executable with a refined grid.


In [ ]:
refined_output = run_command([f"./{PROJECT_NAME}", "2.0"], PROJECT_DIR, timeout=90)
print("refined run completed")
for line in refined_output.splitlines()[:8]:
    if line.strip():
        print(line.rstrip())


## Load Diagnostic Rows

The diagnostic rows provide the numerical evidence used for interpretation.


In [ ]:
diagnostic_rows = {}
for diagnostic in sorted(PROJECT_DIR.glob("out0d-conv_factor*.txt")):
    rows = [
        [float(value) for value in line.split()]
        for line in diagnostic.read_text(
            encoding="utf-8", errors="replace"
        ).splitlines()
        if line.strip()
    ]
    diagnostic_rows[diagnostic.name] = rows
    print(diagnostic.name, "rows:", len(rows), "last row:", rows[-1])


## Compare Errors at Matching Time

The diagnostic rows provide the numerical evidence used for interpretation.


In [ ]:
coarse_name = "out0d-conv_factor1.00.txt"
fine_name = "out0d-conv_factor2.00.txt"
coarse_times = {
    round(row[0], 12) for row in diagnostic_rows[coarse_name] if row[0] > 0.0
}
fine_times = {round(row[0], 12) for row in diagnostic_rows[fine_name] if row[0] > 0.0}
comparison_time = max(coarse_times.intersection(fine_times))
coarse_row = next(
    row for row in diagnostic_rows[coarse_name] if round(row[0], 12) == comparison_time
)
fine_row = next(
    row for row in diagnostic_rows[fine_name] if round(row[0], 12) == comparison_time
)
coarse_error = coarse_row[1]
fine_error = fine_row[1]
print("comparison time:", f"{comparison_time:.6e}")
print("uu error coarse:", f"{coarse_error:.6e}")
print("uu error fine:", f"{fine_error:.6e}")
print("uu error ratio coarse/fine:", coarse_error / fine_error)
if fine_error >= coarse_error:
    raise RuntimeError("Expected the refined run to reduce the uu error.")


The comparison uses diagnostic rows at the same physical time. The smaller refined-grid error is the evidence that the boundary-handled finite-difference run is converging.


## Continue to the Finite-Difference Playground
- [Finite-Difference Playground](finite_difference_playground.ipynb)
- [Curvilinear Boundary Conditions](../4-curvilinear/curvilinear_boundary_conditions.ipynb)
- [Start-to-Finish Cartesian Wave Project](../3-wave_equation/start_to_finish_wave_cartesian.ipynb)
